In [3]:

# =========================================================
# 1. IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    cross_val_score
)

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

import warnings
warnings.filterwarnings("ignore")

In [4]:
# =========================================================
# 2. LOAD DATASET
# =========================================================

data = pd.read_csv("../Data/Energy_consumption_data.csv")



In [5]:
# =========================================================
# 5. FEATURE ENGINEERING FUNCTIONS
# =========================================================

# ---------------------------------------------------------
# Time Period Function
# ---------------------------------------------------------

def get_time_period(hour):

    if 5 <= hour < 12:
        return 0      # Morning

    elif 12 <= hour < 17:
        return 1      # Afternoon

    elif 17 <= hour < 21:
        return 2      # Evening

    else:
        return 3      # Night


# ---------------------------------------------------------
# Weekend Function
# ---------------------------------------------------------

def get_weekend_label(dayofweek):

    return 1 if dayofweek >= 5 else 0

In [6]:
# =========================================================
# 4. DATA PREPROCESSING & CLEANING
# =========================================================

# ---------------------------------------------------------
# Extract Time Features from Timestamp (Fixes KeyError: 'Hour')
# ---------------------------------------------------------
# If your column is named 'timestamp' (lowercase), change 'Timestamp' to 'timestamp' below
if 'Timestamp' in data.columns:
    data['Timestamp'] = pd.to_datetime(data['Timestamp'])
    data['Hour'] = data['Timestamp'].dt.hour
    data['Day'] = data['Timestamp'].dt.day
    data['Month'] = data['Timestamp'].dt.month
else:
    print("⚠️ Warning: 'Timestamp' column not found. Make sure your time column is parsed correctly!")

# ---------------------------------------------------------
# Convert Day Names to Numbers
# ---------------------------------------------------------
day_mapping = {
    "Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3, 
    "Friday": 4, "Saturday": 5, "Sunday": 6
}
data['DayOfWeek'] = data['DayOfWeek'].map(day_mapping)

# ---------------------------------------------------------
# Convert Holiday Column
# ---------------------------------------------------------
holiday_mapping = {"Yes": 1, "No": 0}
data['Holiday'] = data['Holiday'].map(holiday_mapping)

# ---------------------------------------------------------
# Convert HVAC & Lighting
# ---------------------------------------------------------
binary_mapping = {"On": 1, "Off": 0}
data['HVACUsage'] = data['HVACUsage'].map(binary_mapping)
data['LightingUsage'] = data['LightingUsage'].map(binary_mapping)


# =========================================================
# 5. FEATURE ENGINEERING FUNCTIONS
# =========================================================

def get_time_period(hour):
    if 5 <= hour < 12:
        return 0  # Morning
    elif 12 <= hour < 17:
        return 1  # Afternoon
    elif 17 <= hour < 21:
        return 2  # Evening
    else:
        return 3  # Night

def get_weekend_label(dayofweek):
    return 1 if dayofweek >= 5 else 0


# =========================================================
# 6. CREATE ENGINEERED FEATURES
# =========================================================
# Now 'DayOfWeek' and 'Hour' are guaranteed to exist as integers!
data['WeekendLabel'] = data['DayOfWeek'].apply(get_weekend_label)
data['TimePeriodLabel'] = data['Hour'].apply(get_time_period)

In [7]:
data


,Timestamp,Temperature,Humidity,SquareFootage,Occupancy,HVACUsage,LightingUsage,RenewableEnergy,DayOfWeek,Holiday,EnergyConsumption,Hour,Day,Month,WeekendLabel,TimePeriodLabel
0,2022-01-01 00:00:00,25.139433,43.431581,1565.693999,5,1,0,2.774699,0,0,75.364373,0,1,1,0,3
1,2022-01-01 01:00:00,27.731651,54.225919,1411.064918,1,1,1,21.831384,5,0,83.401855,1,1,1,1,3
2,2022-01-01 02:00:00,28.704277,58.907658,1755.715009,2,0,0,6.764672,6,0,78.270888,2,1,1,1,3
3,2022-01-01 03:00:00,20.080469,50.371637,1452.316318,1,0,1,8.623447,2,0,56.519850,3,1,1,0,3
4,2022-01-01 04:00:00,23.097359,51.401421,1094.130359,9,1,0,3.071969,4,0,70.811732,4,1,1,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2022-02-11 11:00:00,28.619382,48.850160,1080.087000,5,0,0,21.194696,5,0,82.306692,11,11,2,1,0
996,2022-02-11 12:00:00,23.836647,47.256435,1705.235156,4,0,1,25.748176,1,1,66.577320,12,11,2,0,1
997,2022-02-11 13:00:00,23.005340,48.720501,1320.285281,6,0,1,0.297079,4,1,72.753471,13,11,2,0,1
998,2022-02-11 14:00:00,25.138365,31.306459,1309.079719,3,1,0,20.425163,3,1,76.950389,14,11,2,0,1


In [8]:
# =========================================================
# 7. FEATURES & TARGET
# =========================================================

features = [

    'Temperature',
    'Humidity',
    'SquareFootage',
    'Occupancy',
    'HVACUsage',
    'LightingUsage',
    'DayOfWeek',
    'Holiday',
    'Hour',
    'Day',
    'Month',
    'WeekendLabel',
    'TimePeriodLabel'

]

target = 'EnergyConsumption'

X = data[features]
y = data[target]



In [9]:
# =========================================================
# 8. TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42

)



# =========================================================
# 9. TRAIN RANDOM FOREST MODEL
# =========================================================

model = RandomForestRegressor(

    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1

)

model.fit(X_train, y_train)



,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [10]:
# =========================================================
# 10. CROSS VALIDATION
# =========================================================

cv_scores = cross_val_score(

    model,
    X,
    y,
    cv=5,
    scoring='r2'

)

print("\n================ CROSS VALIDATION ================")

print("CV Scores:")
print(cv_scores)

print(f"\nAverage CV Score : {cv_scores.mean():.4f}")



================ CROSS VALIDATION ================
CV Scores:
[0.58743039 0.6116006  0.52948501 0.48005589 0.55865999]

Average CV Score : 0.5534


In [26]:
#save the model
import joblib
joblib.dump(model, 'energy_model.pkl')

['energy_model.pkl']

In [11]:
# =========================================================
# 11. PREDICTIONS
# =========================================================

y_pred = model.predict(X_test)


# =========================================================
# 12. MODEL EVALUATION
# =========================================================

r2 = r2_score(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

mae = mean_absolute_error(
    y_test,
    y_pred
)

print("\n================ MODEL PERFORMANCE ================")

print(f"R² Score : {r2:.4f}")
print(f"RMSE     : {rmse:.2f}")
print(f"MAE      : {mae:.2f}")



================ MODEL PERFORMANCE ================
R² Score : 0.5499
RMSE     : 5.43
MAE      : 4.41


In [12]:
# =========================================================
# 13. FEATURE IMPORTANCE
# =========================================================

importance_df = pd.DataFrame({

    'Feature': X_train.columns,
    'Importance': model.feature_importances_

})

importance_df = importance_df.sort_values(

    by='Importance',
    ascending=False

)

print("\n================ FEATURE IMPORTANCE ================")

print(importance_df)





================ FEATURE IMPORTANCE ================
            Feature  Importance
0       Temperature    0.599960
4         HVACUsage    0.077633
3         Occupancy    0.066160
2     SquareFootage    0.055799
1          Humidity    0.053062
9               Day    0.042503
8              Hour    0.039201
6         DayOfWeek    0.025624
5     LightingUsage    0.011620
12  TimePeriodLabel    0.010841
10            Month    0.008220
7           Holiday    0.007262
11     WeekendLabel    0.002114


In [13]:
# =========================================================
# 18. CONSUMER FRIENDLY USER INPUT
# =========================================================

user_input = {

    "PropertyType": "Home",      # Home / Commercial

    "PropertySize": "Small",    # Small / Medium / Large

    "PeopleCount": 10,

    "ACUsage": "Low",           # Low / Medium / High

    "LightingUsage": "Low",   # Low / Medium / High

    "WeatherCondition": "Hot",   # Cool / Normal / Hot

    "Hour": 20,

    "DayOfWeek": 6,

    "Holiday": 1,

    "Day": 29,

    "Month": 5

}


In [14]:
# =========================================================
# 19. BACKEND FEATURE MAPPING
# =========================================================

# ---------------------------------------------------------
# Property Size Mapping
# ---------------------------------------------------------

size_mapping = {

    "Small": 300,
    "Medium": 750,
    "Large": 1000

}

square_footage = size_mapping[
    user_input['PropertySize']
]


# ---------------------------------------------------------
# Weather Mapping
# ---------------------------------------------------------

weather_mapping = {

    "Cool": {
        "Temperature": 22,
        "Humidity": 45
    },

    "Normal": {
        "Temperature": 28,
        "Humidity": 55
    },

    "Hot": {
        "Temperature": 35,
        "Humidity": 65
    }

}

temperature = weather_mapping[
    user_input['WeatherCondition']
]['Temperature']

humidity = weather_mapping[
    user_input['WeatherCondition']
]['Humidity']


# ---------------------------------------------------------
# AC Usage Mapping
# ---------------------------------------------------------

ac_mapping = {

    "Low": 0,
    "Medium": 1,
    "High": 1

}

hvac_usage = ac_mapping[
    user_input['ACUsage']
]


# ---------------------------------------------------------
# Lighting Usage Mapping
# ---------------------------------------------------------

lighting_mapping = {

    "Low": 0,
    "Medium": 1,
    "High": 1

}

lighting_usage = lighting_mapping[
    user_input['LightingUsage']
]


# ---------------------------------------------------------
# Derived Features
# ---------------------------------------------------------

weekend_label = get_weekend_label(
    user_input['DayOfWeek']
)

time_period_label = get_time_period(
    user_input['Hour']
)


In [15]:
# =========================================================
# 20. FINAL MODEL INPUT
# =========================================================

final_input = pd.DataFrame([{

    'Temperature': temperature,

    'Humidity': humidity,

    'SquareFootage': square_footage,

    'Occupancy': user_input['PeopleCount'],

    'HVACUsage': hvac_usage,

    'LightingUsage': lighting_usage,

    'DayOfWeek': user_input['DayOfWeek'],

    'Holiday': user_input['Holiday'],

    'Hour': user_input['Hour'],

    'Day': user_input['Day'],

    'Month': user_input['Month'],

    'WeekendLabel': weekend_label,

    'TimePeriodLabel': time_period_label

}])



In [16]:
# =========================================================
# 21. ENERGY PREDICTION
# =========================================================

predicted_energy = model.predict(final_input)[0]

In [22]:

# =========================================================
# 21. DAILY ENERGY ESTIMATION
# =========================================================

energy_scaling_factor = 0.2

daily_energy_estimate = (
    predicted_energy *
    energy_scaling_factor
) 


# =========================================================
# 22. MONTHLY ENERGY ESTIMATION
# =========================================================

monthly_energy_estimate = daily_energy_estimate *30

In [23]:
# =========================================================
# 24. ELECTRICITY BILL ESTIMATION
# =========================================================

domestic_rate = 6
commercial_rate = 9


if user_input['PropertyType'] == "Home":

    electricity_rate = domestic_rate

else:

    electricity_rate = commercial_rate


daily_bill_estimate = daily_energy_estimate * electricity_rate

monthly_bill_estimate = monthly_energy_estimate * electricity_rate

In [24]:
# =========================================================
# 24. SMART RECOMMENDATION ENGINE
# =========================================================

if monthly_energy_estimate > 1200:

    recommendation = (
        "Very high energy usage detected. "
        "Reduce AC usage and optimize appliance usage."
    )

elif monthly_energy_estimate > 600:

    recommendation = (
        "Moderate energy usage detected. "
        "Monitor lighting and cooling usage."
    )

else:

    recommendation = (
        "Energy usage looks efficient."
    )

In [25]:
# =========================================================
# 25. FINAL OUTPUT
# =========================================================

print("\n================ FINAL RESULT ================")

print(f"\nEstimated Daily Consumption   : {daily_energy_estimate:.2f} kWh")

print(f"\nEstimated Monthly Consumption : {monthly_energy_estimate:.2f} kWh")

print(f"\nEstimated Daily Bill          : ₹{daily_bill_estimate:.2f}")

print(f"\nProjected Monthly Bill        : ₹{monthly_bill_estimate:.2f}")

print(f"\nRecommendation                : {recommendation}")


================ FINAL RESULT ================

Estimated Daily Consumption   : 16.91 kWh

Estimated Monthly Consumption : 507.23 kWh

Estimated Daily Bill          : ₹101.45

Projected Monthly Bill        : ₹3043.39

Recommendation                : Energy usage looks efficient.
